# MoneyPrinterMerged - AI Video Generator
Combined features from MoneyPrinterTurbo + MoneyPrinterV2
**Features:**
- Auto video generation (script + voice + b-roll + subtitles + music)
- Twitter bot with scheduling
- YouTube Shorts automation
- Free b-roll sources (Pexels, Pixabay)
- Mistral AI integration

## Step 1: Clone and Install Dependencies

In [ ]:
!git clone https://github.com/corruptcrew/KyxContent-Hub.git
%cd KyxContent-Hub
!pip install -r requirements.txt -q
!apt-get update -y
!apt-get install -y imagemagick ffmpeg
!sed -i 's/rights="none" pattern="@"/rights="read|write" pattern="@"/' /etc/ImageMagick-6/policy.xml
print('Dependencies installed!')

## Step 2: Configure API Keys

In [ ]:
# Get FREE API keys:# - Pexels: https://www.pexels.com/api/ (free, instant)# - Pixabay: https://pixabay.com/api/ (free, instant)PEXELS_API_KEY = ''PIXABAY_API_KEY = ''MISTRAL_API_KEY = '1frihpaCHeZ09oiXwE8cxLv2PlppFIkj'config_content = f'''[app]video_source = "pexels"pexels_api_keys = ["{PEXELS_API_KEY}"]pixabay_api_keys = ["{PIXABAY_API_KEY}"]llm_provider = "openai"openai_api_key = "{MISTRAL_API_KEY}"openai_base_url = "https://api.mistral.ai/v1"openai_model_name = "mistral-small-latest"subtitle_provider = "edge"voice_name = "en-US-JennyNeural"video_aspect = "portrait"bgm_type = "random"bgm_volume = 0.2listen_host = "0.0.0.0"listen_port = 8080'''with open('config.toml', 'w') as f:    f.write(config_content)print('Configuration saved!')print('Add your Pexels API key above for full video generation')

## Step 3: Start Ngrok Tunnel

In [ ]:
import subprocessimport timeimport requestssubprocess.run(['pkill', '-9', 'ngrok'], stderr=subprocess.DEVNULL)print('Downloading ngrok...')subprocess.run(['wget', '-q', 'https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz', '-O', '/tmp/ngrok.tgz'])subprocess.run(['tar', '-xzf', '/tmp/ngrok.tgz', '-C', '/tmp'])print('Starting ngrok tunnel...')subprocess.Popen(['/tmp/ngrok', 'http', '8501', '--log=stdout'])time.sleep(5)try:    response = requests.get('http://localhost:4040/api/tunnels', timeout=10)    tunnel_url = response.json()['tunnels'][0]['public_url']    print('')    print('=' * 50)    print('SUCCESS!')    print('=' * 50)    print(f'Open this URL in your browser:')    print(f'{tunnel_url}')    print('=' * 50)    print('Keep this cell running while using the WebUI')except Exception as e:    print(f'Check ngrok dashboard: http://localhost:4040')

## Step 4: Start WebUI

In [ ]:
!streamlit run webui/Main.py --server.address=0.0.0.0 --server.port=8501 --server.headless true 2>&1 &time.sleep(8)result = subprocess.run(['lsof', '-i', ':8501'], capture_output=True, text=True)if result.returncode == 0:    print('Streamlit WebUI is running on port 8501')else:    print('Streamlit failed to start')

## Alternative: Generate Video via API

In [ ]:
import requestsimport timefrom google.colab import files!python main.py 2>&1 &time.sleep(5)API_BASE = 'http://localhost:8080'payload = {    'video_subject': 'Which is the best continent?',    'video_aspect': 'portrait',    'video_language': 'en',    'voice_name': 'en-US-JennyNeural',    'voice_volume': 1.0,    'voice_rate': 1.0,    'bgm_type': 'random',    'bgm_volume': 0.2,    'subtitle_enabled': True,    'subtitle_provider': 'edge',    'llm_provider': 'openai'}print('Generating video...')response = requests.post(f'{API_BASE}/api/v1/videos', json=payload)task_id = response.json()['data']['task_id']print(f'Task ID: {task_id}')while True:    status = requests.get(f'{API_BASE}/api/v1/tasks/{task_id}')    data = status.json()['data']    print(f"Progress: {data['progress']}% - State: {data['state']}")        if data['state'] == 1:        print('Video generated!')        video_url = f"{API_BASE}{data['video_url']}"        print(f'Download: {video_url}')                video_response = requests.get(video_url)        with open('video.mp4', 'wb') as f:            f.write(video_response.content)        files.download('video.mp4')        break    elif data['state'] == -1:        print(f"Failed: {data.get('reason', 'Unknown error')}")        break        time.sleep(5)